# Team [42] AI´m lost

### Short Notebook 2

#### Elias Strømsnes - studnummer

#### Hannah Lervik - 590345

#### Kristian Nyland Larsen - 590348

# Imports

In [93]:
import pandas as pd
from prophet import Prophet
from tqdm import tqdm
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

# Load data
### We landed on only using receivals data

In [94]:
receivals = pd.read_csv('data/kernel/receivals.csv', parse_dates=['date_arrival'])
receivals['date_arrival'] = pd.to_datetime(receivals['date_arrival'], utc=True).dt.tz_localize(None)

receivals = receivals[receivals['receival_status'] == 'Completed'].copy()
receivals = receivals[receivals['net_weight'] >= 0].copy()

receivals['date_arrival'] = pd.to_datetime(receivals['date_arrival'])

daily_aggregated = (
    receivals.groupby(['rm_id', pd.Grouper(key='date_arrival', freq='D')])['net_weight']
    .sum().reset_index()
)
print(daily_aggregated)

        rm_id date_arrival  net_weight
0       342.0   2004-06-23     24940.0
1       343.0   2005-03-29     21760.0
2       345.0   2004-09-01     22780.0
3       346.0   2004-06-24       820.0
4       346.0   2004-06-30     21260.0
...       ...          ...         ...
41900  4463.0   2024-10-17      2000.0
41901  4481.0   2024-10-29     24680.0
41902  4481.0   2024-11-27     22340.0
41903  4501.0   2024-12-02     23580.0
41904  4501.0   2024-12-09     24600.0

[41905 rows x 3 columns]


# Prophet + Random Forest Prediction Functions

In [95]:
def create_material_predictions(material_data, material_id=None):
    prophet_df = _prepare_prophet_dataframe(material_data)
    complete_df = _fill_missing_dates(prophet_df)
    enriched_df = _add_time_features(complete_df)
    ensemble_predictions = _create_ensemble_predictions(enriched_df)
    return ensemble_predictions


def _prepare_prophet_dataframe(raw_data):
    return raw_data.copy().rename(columns={
        'date_arrival': 'ds', 
        'net_weight': 'y'
    })


def _fill_missing_dates(prophet_data):
    start_date = prophet_data["ds"].min().normalize()
    end_date = prophet_data["ds"].max().normalize()
    
    complete_date_range = pd.date_range(start_date, end_date, freq="D")
    date_skeleton = pd.DataFrame({"ds": complete_date_range})
    
    return date_skeleton.merge(prophet_data, on="ds", how="left").fillna({"y": 0})


def _add_time_features(base_df):
    enhanced_df = base_df.copy()
    enhanced_df["is_weekday"] = (enhanced_df["ds"].dt.dayofweek < 5).astype(int)
    enhanced_df["day_of_week"] = enhanced_df["ds"].dt.dayofweek
    enhanced_df["month"] = enhanced_df["ds"].dt.month
    enhanced_df["quarter"] = enhanced_df["ds"].dt.quarter
    enhanced_df["lag_7"] = enhanced_df["y"].shift(7)
    enhanced_df["lag_30"] = enhanced_df["y"].shift(30)
    enhanced_df["rolling_mean_7"] = enhanced_df["y"].rolling(window=7, min_periods=1).mean()
    enhanced_df["rolling_mean_30"] = enhanced_df["y"].rolling(window=30, min_periods=1).mean()
    return enhanced_df


def _create_ensemble_predictions(training_data):
    prophet_pred = _get_prophet_predictions(training_data)
    ml_predictions = _get_ml_predictions(training_data)
    ensemble_pred = _combine_predictions(prophet_pred, ml_predictions)
    return ensemble_pred


def _get_prophet_predictions(data):
    try:
        prophet_model = Prophet(
            growth="linear",
            yearly_seasonality=True,
            weekly_seasonality=False,
            daily_seasonality=False,
            seasonality_mode="multiplicative",
            changepoint_prior_scale=0.02,
            changepoint_range=0.8,
            seasonality_prior_scale=0.5,
            holidays_prior_scale=0.56,
        )
        
        prophet_model.add_country_holidays(country_name="NO")
        prophet_model.add_regressor("is_weekday")
        
        prophet_data = data[["ds", "y", "is_weekday"]].copy()
        prophet_model.fit(prophet_data)
        
        future_dates = pd.DataFrame({
            "ds": pd.date_range("2025-01-01", "2025-05-31", freq="D")
        })
        future_dates["is_weekday"] = (future_dates["ds"].dt.dayofweek < 5).astype(int)
        
        forecast = prophet_model.predict(future_dates)
        result = forecast[["ds", "yhat"]].copy()
        result["yhat"] = result["yhat"].clip(lower=0)
        return result
        
    except Exception as e:
        future_dates = pd.DataFrame({
            "ds": pd.date_range("2025-01-01", "2025-05-31", freq="D")
        })
        future_dates["yhat"] = 0
        return future_dates


def _get_ml_predictions(data):
    feature_cols = ["day_of_week", "month", "quarter", "is_weekday", 
                   "lag_7", "lag_30", "rolling_mean_7", "rolling_mean_30"]
    
    ml_data = data.dropna().copy()
    
    if len(ml_data) < 30:
        future_dates = pd.DataFrame({
            "ds": pd.date_range("2025-01-01", "2025-05-31", freq="D")
        })
        return {"rf": future_dates.assign(yhat=0)}
    
    X = ml_data[feature_cols]
    y = ml_data["y"]
    
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    
    model = RandomForestRegressor(n_estimators=20, random_state=42, max_depth=3, 
                                 min_samples_split=20, min_samples_leaf=10, 
                                 max_features='sqrt')
    
    try:
        model.fit(X_scaled, y)
        
        future_features = _generate_future_features(data)
        if len(future_features) > 0:
            future_features_scaled = scaler.transform(future_features[feature_cols])
            future_pred = model.predict(future_features_scaled)
            future_pred = np.clip(future_pred, 0, None)
            
            result = pd.DataFrame({
                "ds": pd.date_range("2025-01-01", "2025-05-31", freq="D"),
                "yhat": future_pred
            })
        else:
            result = pd.DataFrame({
                "ds": pd.date_range("2025-01-01", "2025-05-31", freq="D"),
                "yhat": 0
            })
        
        return {"rf": result}
        
    except Exception as e:
        return {"rf": pd.DataFrame({
            "ds": pd.date_range("2025-01-01", "2025-05-31", freq="D"),
            "yhat": 0
        })}


def _generate_future_features(historical_data):
    future_dates = pd.date_range("2025-01-01", "2025-05-31", freq="D")
    future_df = pd.DataFrame({"ds": future_dates})
    
    future_df["is_weekday"] = (future_df["ds"].dt.dayofweek < 5).astype(int)
    future_df["day_of_week"] = future_df["ds"].dt.dayofweek
    future_df["month"] = future_df["ds"].dt.month
    future_df["quarter"] = future_df["ds"].dt.quarter
    
    if len(historical_data) >= 30:
        lag_7_val = historical_data.tail(14)["y"].median()
        lag_30_val = historical_data.tail(60)["y"].median()
        mean_7_val = historical_data.tail(14)["y"].mean()
        mean_30_val = historical_data.tail(60)["y"].mean()
    else:
        lag_7_val = lag_30_val = mean_7_val = mean_30_val = 0
    
    future_df["lag_7"] = lag_7_val
    future_df["lag_30"] = lag_30_val
    future_df["rolling_mean_7"] = mean_7_val
    future_df["rolling_mean_30"] = mean_30_val
    
    return future_df


def _combine_predictions(prophet_pred, ml_predictions):
    weights = {"prophet": 0.95, "rf": 0.05}
    
    ensemble_pred = prophet_pred[["ds"]].copy()
    
    ensemble_pred["yhat_scaled"] = (
        weights["prophet"] * prophet_pred["yhat"] +
        weights["rf"] * ml_predictions["rf"]["yhat"]
    )
    
    ensemble_pred["yhat_scaled"] = ensemble_pred["yhat_scaled"] * 0.52
    ensemble_pred["yhat_scaled"] = ensemble_pred["yhat_scaled"].clip(lower=0)
    
    return ensemble_pred[["ds", "yhat_scaled"]]

# Process Materials with Prophet+RF

In [96]:
def process_material_forecasting(receivals_data, aggregated_data, min_observations=30):
    all_materials = receivals_data["rm_id"].unique()
    
    valid_materials, excluded_materials = _filter_materials_by_data_quality(
        all_materials, aggregated_data, min_observations
    )
    
    material_forecasts = _batch_generate_predictions(valid_materials, aggregated_data)
    
    consolidated_predictions = _consolidate_prediction_results(material_forecasts)
    
    return consolidated_predictions


def _filter_materials_by_data_quality(material_ids, data, threshold):
    valid_ids = []
    excluded_ids = []
    
    for material_id in tqdm(material_ids, desc="Evaluerer datakvalitet"):
        material_subset = data[data["rm_id"] == material_id]
        
        if len(material_subset) >= threshold:
            valid_ids.append(material_id)
        else:
            excluded_ids.append(material_id)
    
    return valid_ids, excluded_ids


def _batch_generate_predictions(material_list, source_data):
    prediction_collection = []
    
    for current_material in tqdm(material_list, desc="Genererer prediksjoner"):
        try:
            material_timeseries = source_data[source_data["rm_id"] == current_material]
            
            forecast_result = create_material_predictions(material_timeseries, material_id=current_material)
            
            forecast_result = _attach_material_id(forecast_result, current_material)
            
            prediction_collection.append(forecast_result)
            
        except Exception as e:
            fallback_prediction = _create_zero_prediction(current_material)
            prediction_collection.append(fallback_prediction)
    
    return prediction_collection


def _attach_material_id(prediction_df, material_id):
    enhanced_prediction = prediction_df.copy()
    enhanced_prediction["rm_id"] = material_id
    return enhanced_prediction


def _create_zero_prediction(material_id):
    future_dates = pd.DataFrame({
        "ds": pd.date_range("2025-01-01", "2025-05-31", freq="D")
    })
    future_dates["yhat_scaled"] = 0.0
    future_dates["rm_id"] = material_id
    return future_dates


def _consolidate_prediction_results(prediction_list):
    return pd.concat(prediction_list, ignore_index=True)

pred2025 = process_material_forecasting(receivals, daily_aggregated)

Genererer prediksjoner:   0%|          | 0/81 [00:00<?, ?it/s]16:17:13 - cmdstanpy - INFO - Chain [1] start processing
16:17:13 - cmdstanpy - INFO - Chain [1] start processing
16:17:13 - cmdstanpy - INFO - Chain [1] done processing
16:17:13 - cmdstanpy - INFO - Chain [1] done processing
Genererer prediksjoner:   1%|          | 1/81 [00:00<00:17,  4.61it/s]16:17:13 - cmdstanpy - INFO - Chain [1] start processing
16:17:13 - cmdstanpy - INFO - Chain [1] start processing
16:17:13 - cmdstanpy - INFO - Chain [1] done processing
16:17:13 - cmdstanpy - INFO - Chain [1] done processing
Genererer prediksjoner:   2%|▏         | 2/81 [00:00<00:12,  6.41it/s]16:17:14 - cmdstanpy - INFO - Chain [1] start processing
16:17:14 - cmdstanpy - INFO - Chain [1] start processing
16:17:14 - cmdstanpy - INFO - Chain [1] done processing
16:17:14 - cmdstanpy - INFO - Chain [1] done processing
Genererer prediksjoner:   4%|▎         | 3/81 [00:00<00:10,  7.43it/s]16:17:14 - cmdstanpy - INFO - Chain [1] start proc

# Apply Activity Filter & Transform

In [97]:
def apply_recent_activity_filter(prediction_data, historical_data, lookback_days=90):
    analysis_period = _define_analysis_timeframe(lookback_days)
    
    recently_active_materials = _identify_active_materials(historical_data, analysis_period)
    
    filtered_predictions = _zero_out_inactive_predictions(prediction_data, recently_active_materials)
    
    return filtered_predictions


def _define_analysis_timeframe(days_back):
    reference_date = pd.Timestamp('2024-12-31')
    lookback_start = reference_date - pd.Timedelta(days=days_back-1)
    
    return {
        'start_date': lookback_start,
        'end_date': reference_date
    }


def _identify_active_materials(data_source, time_window):
    activity_criteria = (
        (data_source['date_arrival'] >= time_window['start_date']) & 
        (data_source['date_arrival'] <= time_window['end_date']) & 
        (data_source['net_weight'] > 0)
    )
    
    active_material_set = set(data_source.loc[activity_criteria, 'rm_id'].unique())
    
    return active_material_set


def _zero_out_inactive_predictions(predictions, active_material_ids):
    inactive_materials_mask = ~predictions['rm_id'].isin(active_material_ids)
    
    updated_predictions = predictions.copy()
    updated_predictions.loc[inactive_materials_mask, 'yhat_scaled'] = 0.0
    
    return updated_predictions


def transform_to_cumulative_format(predictions_df):
    formatted_data = _standardize_column_names(predictions_df)
    
    enriched_data = _compute_cumulative_weights(formatted_data)
    
    return enriched_data

def _standardize_column_names(data):
    return data.rename(columns={
        "ds": "date", 
        "yhat_scaled": "pred_net_weight"
    })

def _compute_cumulative_weights(data):
    processed_data = data.copy()

    processed_data = processed_data.sort_values(["rm_id", "date"])
    
    processed_data["cum_weight"] = (
        processed_data.groupby("rm_id")["pred_net_weight"].cumsum()
    )
    
    return processed_data

pred2025_filtered = apply_recent_activity_filter(pred2025, daily_aggregated)
pred2025cum = transform_to_cumulative_format(pred2025_filtered)

# Create Prophet+RF Submission

In [98]:
def create_final_submission(predictions_data, mapping_file="data/prediction_mapping.csv", output_file="submission_ensemble.csv"):
    clean_predictions, clean_mapping = _prepare_datasets_for_merge(predictions_data, mapping_file)

    combined_data = _merge_predictions_with_mapping(clean_predictions, clean_mapping)
    
    final_submission = _format_submission_output(combined_data)
    
    _save_submission_file(final_submission, output_file)
    
    return final_submission


def _prepare_datasets_for_merge(predictions, mapping):
    mapping_data = pd.read_csv(mapping)

    clean_pred = predictions.copy()
    clean_pred["date"] = pd.to_datetime(clean_pred["date"])
    clean_pred["rm_id"] = clean_pred["rm_id"].astype(float)
    
    clean_map = mapping_data.copy()
    clean_map["forecast_end_date"] = pd.to_datetime(clean_map["forecast_end_date"], errors="coerce")
    clean_map["rm_id"] = clean_map["rm_id"].astype(float)
    
    return clean_pred, clean_map


def _merge_predictions_with_mapping(predictions, mapping):
    merged_result = mapping.merge(
        predictions,
        left_on=["rm_id", "forecast_end_date"],
        right_on=["rm_id", "date"],
        how="left"
    )
    
    merged_result["cum_weight"] = merged_result["cum_weight"].fillna(0)
    
    return merged_result


def _format_submission_output(merged_data):
    submission_data = merged_data[["ID", "cum_weight"]].copy()
    submission_data = submission_data.rename(columns={"cum_weight": "predicted_weight"})
    
    return submission_data

def _save_submission_file(data, filename):
    data.to_csv(filename, index=False)

submission = create_final_submission(pred2025cum)